# load_openalex_work_location

Prototipo del nodo `load_openalex_work_location` del pipeline `load_openalex`. No guarda datasets.


In [ ]:
import pandas as pd
from pandas import json_normalize

%load_ext kedro.ipython


In [ ]:
df_work_raw = catalog.load('raw/openalex/work/parquet/work_dev')
df_work_raw.head(2)


In [ ]:
def _select_with_metadata(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = _add_openalex_extracted_metadata(df)
    return df.loc[:, [*columns, *_EXTRACTED_META_COLS]].copy()


In [ ]:
def _stringify_object_columns(
    df: pd.DataFrame,
    exclude_columns: list[str] | None = None,
) -> pd.DataFrame:
    exclude_columns = set(exclude_columns or [])
    for column in df.columns:
        if column in exclude_columns:
            continue
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].where(df[column].notna(), pd.NA).astype("string")
    return df


In [ ]:
def _serialize_nested_value(value):
    if value is None or value is pd.NA:
        return value
    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


In [ ]:
def _serialize_nested_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(_serialize_nested_value)
    return df


In [ ]:
def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df["extract_datetime"] = pd.to_datetime(df["extract_datetime"], errors="coerce")
    df["_extract_datetime"] = pd.to_datetime(df["_extract_datetime"], errors="coerce")
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date
    return df


In [ ]:
def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = _serialize_nested_columns(df)
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    load_datetime = pd.to_datetime(load_datetime)
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openalex_work_location(df_work_raw, load_datetime=None):

    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    df_work_location = df_work_raw.explode('locations').reset_index(drop=True)

    df_locations = json_normalize(df_work_location['locations'])
    df_locations.rename(columns={'id': 'location_id'}, inplace=True)

    df_work_location = pd.concat(
        [df_work_location.loc[:, ['id', *_EXTRACTED_META_COLS]], df_locations],
        axis=1,
    )

    df_work_location.columns = df_work_location.columns.str.replace('.', '_')

    df_work_location = df_work_location[[
        'id',
        'source_id', 'source_display_name', 'source_is_core', 'source_type',
        'source_host_organization', 'source_host_organization_name',
        'is_accepted', 'is_oa', 'is_published', 'landing_page_url',
        'license', 'license_id', 'pdf_url', 'version',
        # 'source_host_organization_lineage', 'source_host_organization_lineage_names', 'source_issn',
        'source_is_in_doaj', 'source_is_oa', 'source_issn_l',
        *_EXTRACTED_META_COLS,
    ]]

    df_work_location = _add_openalex_loaded_metadata(df_work_location, load_datetime=load_datetime)

    return df_work_location


In [ ]:
df_work_location = load_openalex_work_location(df_work_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_work_location', 'rows': len(df_work_location), 'columns': len(df_work_location.columns)}])


In [ ]:
df_work_location.head(2)
